# Bermuda Python Environment

This is a Python environment you can use for data analysis on the Oxford Earth Sciences Bermuda field course. The following packages are available:

* `numpy`
* `scipy`
* `matplotlib`
* `pandas`
* `gsw`
* `PyCO2SYS`

The following code block also contains some helper functions that might be useful.

In [ ]:
import pandas as pd
from gsw.conversions import SP_from_C
from io import StringIO

def read_ctd(fh, return_metadata=True):
    
    '''
    This function imports data from the Valeport miniCTD raw text file output, and calculates
    salinity (PSU) from conductivity (mS/cm).

    Output: [Pandas dataframe, dict with metadata]
    '''

    # Open file and process metadata first
    metadata = {}
    data_lines = []
    
    with open(fh, 'r') as f:
        reading_data = False
    
        for line in f:
            stripped = line.strip() # Remove whitespace

            # Skip blank lines
            if not stripped:
                continue
    
            if not reading_data and ':' in stripped:
                # Metadata (all metadata lines have a colon)
                key, value = stripped.split(':', 1)
                metadata[key.strip()] = value.strip()
            else:
                # Numerical data starts here
                reading_data = True
                data_lines.append(line)

    # Write units to metadata
    metadata['P_units'] = 'dbar'
    metadata['T_units'] = 'degC'
    metadata['S_units'] = 'PSU'
    
    # Convert numerical section to DataFrame
    df = pd.read_csv(
        StringIO(''.join(data_lines)),
        sep='\t',
        header=None,
        names=['P', 'T', 'C'])

    # Convert conductivity to salinity
    df['S'] = SP_from_C(df['C'], df['T'], df['P'])
    df = df[['P', 'T', 'S']] # Remove conductivity

    if return_metadata:
        return df, metadata
    else:
        return df

In [ ]:
# Now you can write your own code...
# Example:

import matplotlib.pyplot as plt
import numpy as np

data, metadata = read_ctd('data/ctd/example_dataset.TXT') # Get CTD data
data = data[data['P'] > 0.1] # Remove data with P < 0.1 dbar (surface)
smin, smax = np.quantile(data['S'], 0.1), np.quantile(data['S'], 0.9)

# Create a simple plot
f, ax = plt.subplots(1, 1, figsize=(4, 6))
plot = ax.scatter(data['T'], data['P'], c=data['S'], marker='.', s=1,
                  vmin=smin, vmax=smax) # Color points by salinity
ax.yaxis.set_inverted(True)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.set_xlabel('Temperature (C)')
ax.set_ylabel('Pressure (dbar)')
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
plt.colorbar(plot, label='Salinity (PSU)')
